# Bible AI Fine-Tuning Notebook (Kaggle)
## Trains a small model on KJV + Strong's + Commentaries

### Before running:
1. Go to **Settings (right panel) → Accelerator → GPU T4 x2** (free)
2. Enable **Internet** in Settings (required to download the base model)
3. Add your `training_data.jsonl` via **+ Add Data → Upload Dataset** (right panel)
4. Run each cell in order

**Total time: ~1-2 hours on free Kaggle GPU**

> 💡 Kaggle gives **30 hrs/week** of free GPU — sessions don't time out mid-run like Colab.

## Step 1 — Install Unsloth and dependencies

In [ ]:
# Install datasets and trl first, then unsloth last so it pulls
# a compatible version of transformers automatically
!pip install datasets trl -q
!pip install unsloth -q
print('✅ Installation complete')

## Step 2 — Load the base model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Model options (choose one):
# 'unsloth/llama-3.2-1b-instruct'   <- recommended, 1B params, good quality
# 'unsloth/qwen2.5-0.5b-instruct'   <- smallest, fastest, less capable
# 'unsloth/llama-3.2-3b-instruct'   <- better quality, needs more RAM

MODEL_NAME = 'unsloth/llama-3.2-1b-instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,   # saves memory
    dtype = None,
)

print(f'✅ Loaded {MODEL_NAME}')

## Step 3 — Apply LoRA (efficient fine-tuning adapter)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank — higher = more learning, more memory
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)

print('✅ LoRA adapter applied')

## Step 4 — Load your training data

In [ ]:
import json
import glob
from datasets import Dataset

# Auto-find the training_data.jsonl file anywhere under /kaggle/input/
# This works regardless of your dataset folder name
matches = glob.glob('/kaggle/input/**/training_data.jsonl', recursive=True)
if not matches:
    # Fallback: find any .jsonl file
    matches = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
if not matches:
    raise FileNotFoundError(
        'No .jsonl file found in /kaggle/input/\n'
        'Make sure you added your dataset via + Add Data on the right panel.'
    )

TRAINING_DATA_PATH = matches[0]
print(f'✅ Found training data: {TRAINING_DATA_PATH}')

# The system prompt that defines your AI's behavior
SYSTEM_PROMPT = """You are a Bible reference assistant. You answer questions strictly 
from the KJV Bible, Strong's Concordance, and historical Bible commentaries. 
You always cite the book, chapter, and verse. You use Strong's Concordance to 
understand the original Hebrew and Greek meaning of words when relevant. 
You never make up Bible verses or generate devotions, prayers, or sermons. 
If a question is not answerable from scripture and commentaries, you say: 
'I can only answer from the KJV Bible, Strong's Concordance, and historical 
commentaries. I cannot help with that request.'"""

def format_prompt(instruction, response):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>
{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{response}<|eot_id|>"""

# Load the JSONL file
data = []
with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        text = format_prompt(item['instruction'], item['response'])
        data.append({'text': text})

dataset = Dataset.from_list(data)

print(f'✅ Loaded {len(dataset):,} training examples')
print(f'Sample prompt preview:')
print(dataset[0]['text'][:300])

## Step 5 — Configure training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Kaggle output directory — all files saved here are downloadable after the run
OUTPUT_DIR = '/kaggle/working/bible_model_output'

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 2,        # run through all data 2 times
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 25,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = OUTPUT_DIR,
        report_to = 'none',
    ),
)

print('✅ Trainer configured')
print(f'   Training on {len(dataset):,} examples')
print(f'   2 epochs = ~{len(dataset)*2:,} steps total')

## Step 6 — Train! (this takes ~1-2 hours)

In [ ]:
print('🚀 Starting training...')
print('You will see loss numbers — they should decrease over time')
print('Lower loss = better learning')
print()

trainer_stats = trainer.train()

print()
print('✅ Training complete!')
print(f'   Time: {trainer_stats.metrics["train_runtime"]/60:.1f} minutes')
print(f'   Final loss: {trainer_stats.metrics["train_loss"]:.4f}')

## Step 7 — Test your model

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,      # low temp = more factual, less creative
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant response
    if 'assistant' in result:
        result = result.split('assistant')[-1].strip()
    print(f'Q: {question}')
    print(f'A: {result}')
    print()

# Test with Bible questions
ask('What does John 3:16 say?')
ask('What does the Bible say about fear?')
ask('What is the Hebrew meaning of shalom?')

# Test the filter
ask('Write me a devotion for today.')
ask('What is the weather today?')

## Step 8 — Save the model (download to your computer)

In [ ]:
# Save in GGUF format — this is the format Ollama uses to run on your CPU
# All output is saved to /kaggle/working/ which is downloadable from the Output tab
print('Saving model in GGUF format for Ollama...')

GGUF_OUTPUT_DIR = '/kaggle/working/bible_ai_model'

model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',   # good balance of size and quality
)

print('✅ Saved!')
print()
print(f'Your model is at: {GGUF_OUTPUT_DIR}/bible_ai_model-Q4_K_M.gguf')
print()
print('To download:')
print('  1. Click the Output tab (top right of the notebook)')
print('  2. Navigate to bible_ai_model/')
print('  3. Download bible_ai_model-Q4_K_M.gguf')

## Step 9 — Run on your CPU with Ollama

After downloading the `.gguf` file to your computer:

1. Install Ollama from **ollama.com**

2. Create a file called `Modelfile` with this content:
```
FROM ./bible_ai_model-Q4_K_M.gguf
SYSTEM "You are a Bible reference assistant. You answer questions strictly from the KJV Bible, Strong's Concordance, and historical Bible commentaries. You always cite the book, chapter, and verse. You use Strong's Concordance to understand the original Hebrew and Greek meaning of words when relevant. You never make up Bible verses or generate devotions, prayers, or sermons. If a question is not answerable from scripture and commentaries, you say: I can only answer from the KJV Bible, Strong's Concordance, and historical commentaries. I cannot help with that request."
```

3. Open terminal in the same folder and run:
```
ollama create bible-ai -f Modelfile
ollama run bible-ai
```

4. Start asking Bible questions!
